# Train the image captioner on a free Colab GPU

Runtime → Change runtime type → **T4 GPU**. Roughly: downloads ~30 min, feature extraction ~20 min,
cross-entropy training 2-4 h, SCST 1-3 h. Checkpoints are saved to Google Drive, so if the session
disconnects just re-run all cells and training resumes where it stopped.

Disk: images ~19 GB + CLIP ViT-B/16 features ~37 GB fits on Colab's local disk. If space runs out,
use `ENCODER = "clip-vit-b32"` (~10 GB of features, slightly lower quality).

In [ ]:
ENCODER = "clip-vit-b16"
DRIVE_DIR = "/content/drive/MyDrive/image-captioner"   # checkpoints + results are kept here

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!git clone https://github.com/sarvesh-raam/image-captioning.nlp.git /content/image-captioner
%cd /content/image-captioner
!pip install -q -r requirements-train.txt
!mkdir -p "$DRIVE_DIR/checkpoints" "$DRIVE_DIR/results"
!ln -sfn "$DRIVE_DIR/checkpoints" checkpoints
!ln -sfn "$DRIVE_DIR/results" results
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Download COCO 2014 images and build the Karpathy split

In [ ]:
!mkdir -p coco_images
%cd coco_images
!wget -q --show-progress http://images.cocodataset.org/zips/train2014.zip
!wget -q --show-progress http://images.cocodataset.org/zips/val2014.zip
!unzip -q train2014.zip && rm train2014.zip
!unzip -q val2014.zip && rm val2014.zip
%cd /content/image-captioner
!python prepare_data.py --only-local

## 2. Cache backbone features (run once)

In [ ]:
!python extract_features.py --encoder $ENCODER --batch-size 128 --workers 2
# The raw images are no longer needed for training
!rm -rf coco_images/train2014
!df -h /content | tail -1

## 3. Cross-entropy training (resumes automatically)

In [ ]:
!python train.py --encoder $ENCODER

## 4. Self-critical sequence training (optimises CIDEr)

In [ ]:
!python train.py --encoder $ENCODER --scst

## 5. Evaluate on the Karpathy test split

In [ ]:
!python evaluate.py --checkpoint checkpoints/{ENCODER}_xe_best.pth --name {ENCODER}_xe
!python evaluate.py --checkpoint checkpoints/{ENCODER}_scst_best.pth --name {ENCODER}_scst --beam-sizes 1 3 5 --ablate-penalty
!cat results/{ENCODER}_xe.md results/{ENCODER}_scst.md

## 6. Export and upload to the Hugging Face Model Hub

In [ ]:
!python export_model.py checkpoints/{ENCODER}_scst_best.pth checkpoints/model.pth

from getpass import getpass
from huggingface_hub import HfApi
api = HfApi(token=getpass("Hugging Face WRITE token: "))
api.upload_file(path_or_fileobj="checkpoints/model.pth", path_in_repo="model.pth",
                repo_id="sarveshraam/image-captioner-model", repo_type="model")
print("Uploaded. Now restart / rebuild the Space so it picks up the new model.")